In [1]:
# importing libraries
import os # environment setup
from langchain_groq import ChatGroq  # llm model to be used
from dotenv import load_dotenv # to load api keys to establish connections in hidden document
from typing import List, Tuple
from transformers import AutoTokenizer

load_dotenv() # loading the api key and dependencies

llm = ChatGroq(model = "llama-3.3-70b-versatile", 
               temperature= 2)

#### **important note**
In the .env file ly groq api is mentioned as GROQ_API_KEY, and this is default, I dont need to explicitly mention it and the program will find it out on it's own

In [2]:
## input file path

file_path =  r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\memo_2_test.docx"

### **Data Processing**

#### **Loading Data**

In [3]:
## functions for reading different file formats

import os
import fitz
from docx import Document

# for .txt files
def read_txt(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()
    
# for .docx files
def read_docs(file_path):
    document = Document(file_path)
    full_text = []
    
    for paragraph in document.paragraphs:
        full_text.append(paragraph.text)
    return '\n'.join(full_text)


# for .pdf files

def read_pdf(file_path):
    document = fitz.open(file_path)
    all_text = []
    for page in document:
        text = page.get_text()
        all_text.append(text)
    return '/n'.join(all_text)

In [4]:
# reading the files

def read_file(file_path):
    
    # checking if file exist
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f'The file {file_path} does not exist.')
    
    # getting the file extension
    _, file_extension = os.path.splitext(file_path)

    # handles .txt file
    if file_extension.lower() == '.txt':
        return read_txt(file_path)
    
    # handles .docx file
    elif file_extension.lower() == '.docx':
        return read_docs(file_path)
    
    # handles .docx file
    elif file_extension.lower() == '.pdf':
        return read_pdf(file_path)
    
    else:
        raise ValueError(f"Unsupported file extension: {file_extension}")

In [5]:
# reading the file

text = read_file(file_path)
print(text[:500])

Input:
I would like to seek expert advice regarding our company's upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?
Follow up question: What is the volume of the transactions?
Follow up question: What is the date of the transactions



#### **Processing the data**

In [6]:
# re is used for using regular expression for advanced pattern matching
import re

In [7]:
# text segmentation

def text_segmentation(text: str) -> List[str]:
    """
    Split text into smaller, manageable chunks with consideration for legal memo structures.

    Args:
        text (str): The full text to be segmented.

    Returns:
        List[str]: A list of segmented text chunks.
    """

    # identify the section breaks
    patterns = r'(?<=\n)(?=\n)|(?<=\n)(?=\s*[\d-]+\s)|(?<=\n)(?=Section \d+|Article \d+)|(?<=\n)(?=\s*-\s)|(?<=\n)(?=\s*\*\s)'

    # split the texts using defined patterns
    segments = re.split(patterns, text) #.split())

    # clean up the space to remove any leading or trailing whitespaces
    segment_text = [segment.strip() for segment in segments if segment.strip()]

    return segment_text

# segmented_text = text_segmentation(text)
# segmented_text

# Why Use NLTK with Word Embeddings in Legal RAG?

In a modern RAG (Retrieval-Augmented Generation) pipeline using models like **Llama-3**, the use of "old school" NLP techniques like **Stopword Removal** and **Lemmatization** might seem redundant. However, in the **Legal Domain**, these steps serve specific strategic purposes.



## 1. Hybrid Search (The "Golden Standard")
Most professional RAG systems do not rely solely on **Semantic Search** (Embeddings). They use **Hybrid Search**, which combines two different engines:

* **Vector Search (Dense):** Understands meaning. If you search for *"attorney fees,"* it finds documents about *"legal costs."*
* **Keyword Search (Sparse/BM25):** Finds exact matches. If you search for a specific Case ID like *"21-CV-456,"* vector embeddings might struggle, but keyword search will find it instantly.

**NLTK's Role:** Keyword search engines (like BM25) perform significantly better when text is **lemmatized** (reducing "arguing" and "argued" to "argue"), as it increases the "hit rate" for specific legal terms.

---

## 2. Noise Reduction in "Legalese"
Legal documents are filled with high-frequency "filler" words that carry little semantic weight but appear constantly:
> *Ex: "hereinafter," "aforementioned," "whereas," "therein."*

* **The Problem:** These words occupy space in the **Embedding Model's** limited context window.
* **The NLTK Fix:** By removing these during the indexing phase, you ensure the model focuses its mathematical "attention" on the core legal facts (the "theft," the "breach," the "liability") rather than the formal fluff.

---

## 3. Cost and Context Efficiency
Legal cases are often hundreds of pages long. 
* **Original Text:** 1,000 tokens.
* **NLTK Cleaned Text:** ~400 tokens.

By stripping stopwords and lemmatizing, you can fit **more relevant chunks** from different cases into a single prompt for Llama-3, providing the AI with a broader knowledge base for its final answer.

---

## 4. When to AVOID NLTK (The Risks)
While NLTK is great for "search," it can be dangerous for "understanding":

* **Loss of Tense:** In law, *"The defendant **is** liable"* is very different from *"The defendant **was** liable."* Lemmatization turns both into *"defendant be liable."*
* **Loss of Nuance:** Removing the word "not" or "no" (standard stopwords) can completely flip the meaning of a legal clause.



### ✅ Best Practice Recommendation:
1.  **Use NLTK-cleaned text** for your **Keyword/BM25 Index** (to help find the right document).
2.  **Use RAW text** for your **Vector Embeddings** and the final **LLM Prompt** (to ensure the AI understands the full context and legal nuances).

In [8]:
# Text preprocessing pipeline
# here tokens are created for meaning
"""
    it takes raw input and processes it so that the machine can understand it
"""

import re
from typing import List
import nltk
from nltk.tokenize import word_tokenize # breaks long string of texts into words
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure nltk resources are downloaded during setup or first run
def setup_nltk():
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True) # rules for where sentences and word end
    nltk.download('wordnet', quiet=True)

setup_nltk()

def tokenize_text(text: str)-> List[str]:
    """
    Tokenize the text into words, considering legal-specific tokens and preprocessing.

    Args:
        text (str): The text to be tokenized.

    Returns:
        List[str]: A list of tokens (words).
    """
    # # Remove unwanted characters, retain alphanumeric and some punctuation relevant to legal text
    # cleaned_text = re.sub(r'[^\w\s.,;:()\'\"-]', ' ', text)

    # Tokenize using NLTK's word tokenizer, which handles punctuation better than simple regex
    tokens = word_tokenize(text)

    # convert to lowercase to main consistancy
    tokens = [token.lower() for token in tokens]

    # remove stopwords specific to legal context if needed
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]

    # lemmatize tokens to reduce them to their base form 
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return tokens

In [9]:
# removing special characters

def clean_special_characters(text: str)-> str:
    """
    Clean up non-informative special characters or artifacts.

    Args:
        text (str): The text from which to remove special characters.

    Returns:
        str: Cleaned text with unnecessary special characters removed.
    """

    # remove characters not usually found in legal texts
    cleaned_text = re.sub(r'[^\w\s,.!?;:()-]', '', text)

    return cleaned_text

In [10]:
def preserve_structure(text: str) -> str:
    """
    Maintain the document's structural elements, such as headings.

    Args:
        text (str): The text to process for structural preservation.

    Returns:
        str: Text with preserved structure for headings and sections.
    """
    # Keep lines starting with capital words as headings
    structured_text = re.sub(r'(?m)^(?=[A-Z])(.+)$', r'## \1', text)

    return structured_text

In [11]:
# here we tokenize for size to cut up the text for smaller size

def create_chunks(text: str, tokenizer, chunk_size = 4096) -> List[str]:
    """
    Creates chunks of text for preprocessing, ensuring each chunk is within the specified size.

    Args:
        text (str): The full text to be chunked.
        tokenizer (object): The tokenizer object to encode and decode text.
        chunk_size (int): The desired size of each chunk.

    Returns:
        List[str]: A list of text chunks.
    """

    # tokenize the text
    tokens = tokenizer.encode(text, add_special_tokens = True, truncation = True, max_length =chunk_size)

    # split tokens into chunks
    chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]

    # decode each chunk back into text
    text_chunks = [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]

    return text_chunks

### Token-Based Chunking Logic

This section ensures our legal documents are split into "AI-ready" pieces. 

1. **Encoding**: Text is converted into Token IDs (the numerical language of LLMs).
2. **Slicing**: The list of IDs is sliced into segments of exactly `chunk_size` (e.g., 4096). This prevents "Context Window" errors.
3. **Decoding**: The segments are converted back into strings.

**Benefit**: This is more accurate than character-counting because 1,000 characters does not always equal 1,000 tokens. By chunking on the **tokens** themselves, we maximize the amount of information we can store in each vector without overflowing the model's limit.

### Comparison: Slicing vs. Cleaning

In this RAG pipeline, we use two different types of "tokenization" at different stages:

#### 1. Structural Chunking (`create_chunks`)
* **Purpose:** To stay within the LLM's **Context Window** (e.g., 4096 tokens).
* **Logic:** Uses a model-specific tokenizer (HuggingFace/Tiktoken) to split text into manageable blocks.
* **Usage:** These chunks are what we **embed** and store in our Vector Database.

#### 2. Linguistic Preprocessing (`tokenize_text`)
* **Purpose:** To optimize **Keyword Search** (BM25).
* **Logic:** Uses **NLTK** to lowercase, remove stopwords, and lemmatize (e.g., "Argued" -> "Argue").
* **Usage:** This processed list of words helps the search engine find exact legal matches regardless of grammar.

**Relationship:** We usually **Chunk** the text first, and then we **Tokenize/Clean** the text inside those chunks to build our search index.

In [12]:
def preprocess_doc(file_path: str, tokenizer, chunk_size = 4096, overlap = 0)-> Tuple[str, List[str], str, List[str], str, List[str]]:

    """
    Preprocess a legal document by executing a series of text processing steps, including chunking.

    Args:
        file_path (str): The path to the legal document text file.
        tokenizer (object): The tokenizer object to encode and decode text.
        chunk_size (int): The desired size of each chunk.
        overlap (int): The number of tokens to overlap between chunks.

    Returns:
        Tuple: A tuple containing:
            - Original text (str)
            - Segmented text chunks (List[str])
            - Cleaned text (str)
            - Tokenized words (List[str])
            - Structured text (str)
            - Chunks (List[str])
    """

    # load text from the file
    text = read_file(file_path)

    # split the text into segments(paragraphs)
    segments = text_segmentation(text)

    # Clean up non-informative special characters
    cleaned_text = clean_special_characters(text)

    # Tokenize the text into words
    tokens = tokenize_text(cleaned_text)

    # Preserve structural elements (headings)
    structured_text = preserve_structure(cleaned_text)

    # Create chunks
    chunks = create_chunks(cleaned_text, tokenizer, chunk_size)

    return text, segments, cleaned_text, tokens, structured_text, chunks

### Preprocessing Execution: From Raw File to Model-Ready Data

This block initializes the **Longformer Tokenizer** and executes the full preprocessing pipeline (`preprocess_doc`). 

#### Key Components:
* **Longformer (4096):** Chosen for its large context window, allowing us to process long legal sections without losing context.
* **Variable Unpacking:** The pipeline returns six different versions of the data, ranging from raw text (for humans) to lemmatized tokens (for keyword search) and encoded chunks (for the Vector DB).

#### Output Verification:
The print statements serve as a debug layer to ensure the text has been correctly:
1. **Extracted** from the source file.
2. **Restructured** with Markdown headings.
3. **Cleaned** of stopwords and lemmatized.
4. **Chunked** into exactly 4096-token segments for the model.

In [13]:
# this is for execution pipeline 
# using preprocessor
from transformers import LongformerTokenizer

# initiate longformer tokenizer
tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')

# call the preproess_doc function
original_text, segments, cleaned_text, tokens, structured_text, chunks = preprocess_doc(file_path, tokenizer)

print("Original Text:", original_text[:50], "...")
print("---" * 25)
print("Segments:", segments[:1])
print("---" * 25)
print("Cleaned Text:", cleaned_text[:50], "...")
print("---" * 25)
print("Tokens:", tokens[:5])
print("---" * 25)
print("Structured Text:", structured_text[:50], "...")
print("---" * 25)
print("Chunks:", chunks[:1])

Original Text: Input:
I would like to seek expert advice regardin ...
---------------------------------------------------------------------------
Segments: ["Input:\nI would like to seek expert advice regarding our company's upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?\nFollow up question: What is the volume of the transactions?\nFollow up question: What is the date of the transactions\nFollow up question: What was the price of purchased ETH?/ What was the value of ETH expressed in stablecoin?\nFollow up question: What is the stablecoin pegged to? \nFollow up question: Has this transaction will happen between dependent or independent parties?\nFollow up question: When is the transaction planned for?\nFollow up question

### Comparison: Full Pipeline vs. Size Inspection

While both code blocks utilize the `LongformerTokenizer`, they serve different roles in the RAG architecture:

| Feature | `preprocess_doc` (Pipeline) | `tokenizer()` (Inspection) |
| :--- | :--- | :--- |
| **Primary Goal** | Prepares text for storage/retrieval. | Validates token counts and tensor shapes. |
| **Output Type** | List of Strings (Readable). | PyTorch Tensor (Numerical). |
| **Truncation** | Strategic (handled by `create_chunks`). | Hard-stop (forced by `max_length=4096`). |
| **Use Case** | Running the actual application. | Debugging and performance tuning. |

**Summary:** The `preprocess_doc` function is the **"What"** (the data we are using), while the manual tokenizer check is the **"How Much"** (verifying the data fits the model's constraints).

In [14]:
# this is validation
# tokenize the cleaned_text to inspect the embedding size
sampled_tokenized = tokenizer(cleaned_text, return_tensors= 'pt', max_length=4096, truncation=True)

# print the size of the embedding
embedding_size = sampled_tokenized['input_ids'].size(1) #gets the number of tokens
print(f'embedding size : {embedding_size}')

embedding size : 1517


### Token Count Validation (Pre-Embedding Check)

Before sending our text to Pinecone or OpenAI, we must verify its "token length." 

**Why this matters:**
* **Model Limits:** Our Longformer model has a 4096-token ceiling. 
* **Data Integrity:** We need to ensure the `create_chunks` function didn't accidentally lose text or create "over-sized" blocks that will be truncated (cut off) during the embedding process.
* **Cost Tracking:** Token counts directly correlate to API costs when using providers like OpenAI or Groq.

**The Output:**
The `embedding_size` printed below represents the number of tokens the model "sees." We aim for this to be $\leq 4096$ to ensure no data loss occurs.

### Interpreting the Validation Result

The output `embedding size: 1517` confirms two important things:

1. **Safety:** Our text is well within the **4096 token limit** of the Longformer model. We are using ~37% of the available context window, meaning **zero data loss** occurred during tokenization.
2. **Efficiency:** The 1517 tokens represent roughly 1,100 words of "dense" legal information. After cleaning (removing stopwords and lemmatization), we have a highly optimized input ready for the vector database.

**Capacity (4096)** is the maximum allowed; **Result (1517)** is our actual usage.

### Setting up the vector database

In [15]:
# import os
# from dotenv import load_dotenv
# only import chromadb
import chromadb
# from groq import Groq no need as we have already imported groq using langchain earlier

# load environment varible
# load_dotenv()

# 3. Initialize ChromaDB (Persistent Local Storage)
# This path is perfect for Hugging Face Spaces
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# create or get the collection 
collection_name = "Ishank_legal_docs"
collection = chroma_client.get_or_create_collection(name= collection_name, metadata={"hnsw:space": "cosine"})


print(f"ChromaDB initialized. Collection '{collection_name}' is ready.")
print(f"LangChain Groq (Llama 3.3) is ready: {llm is not None}")


ChromaDB initialized. Collection 'Ishank_legal_docs' is ready.
LangChain Groq (Llama 3.3) is ready: True


### Vector Database and API Configuration

This block sets up the external infrastructure required for our Legal RAG system.

#### Key Infrastructure Components:
* **Pinecone Index:** Named `luthor-test-nb-0`, acting as our long-term semantic memory.
* **Vector Dimensions (1536):** Configured specifically to match the output of OpenAI's embedding models.
* **Similarity Metric (Cosine):** Optimized for finding semantic relationships between legal queries and document chunks.
* **Cloud Provider:** Hosted on **AWS (us-east-1)** using a Serverless specification for cost-efficiency.

#### Workflow:
1. Load secrets from `.env`.
2. Authenticate with Pinecone.
3. Conditionally create the index if it does not already exist.
4. Authenticate with OpenAI to prepare for text-to-vector transformation.

### Transitioning to Open Source: ChromaDB & Groq

We have shifted our infrastructure to be more portable and cost-effective for deployment on **Hugging Face Spaces**.

#### Changes:
1. **Vector Store:** Swapped Pinecone for **ChromaDB**. This allows us to store our vector index locally within the app's container.
2. **Inference Engine:** Swapped OpenAI for **Groq**, enabling us to use **Llama-3** with extremely low latency.
3. **Deployment Strategy:** Configured for HF Spaces using `PersistentClient` for local file-based storage.

#### Deployment Checklist:
* Added `GROQ_API_KEY` to Hugging Face Secrets.
* Included `chromadb` and `groq` in `requirements.txt`.
* Set up a local Embedding Function since Groq focuses on Chat completion rather than Vectorization.

In [17]:
# embedding the converted chunks

from langchain_huggingface import HuggingFaceEmbeddings # for embedding the chunks
from langchain_chroma import Chroma

# the embedding model
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

# 2. Add your chunks to ChromaDB (It handles the embedding process for you!)
# 'chunks' should be the list of strings from your preprocessing
vectorstore = Chroma.from_texts(
    texts = chunks,
    embedding = embedding_model,
    persist_directory = "./chroma_db"
)

# verify
print(f"Success! {len(chunks)} chunks have been embedded and stored in ChromaDB.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success! 1 chunks have been embedded and stored in ChromaDB.


In [ ]:
# the search engine part of the rag
# it takes human question, converts it into vectors and asks chromadb database
# to find out the most similar piece of text it has stored

In [19]:
# 1. Define your question
question = "What are the essential elements of a contract in English law?"

# 2. Query ChromaDB directly
# Chroma handles the embedding of the 'question' automatically using the model you provided earlier.
# 'k' is the number of matches you want (like top_k in Pinecone)
matches = vectorstore.similarity_search(question, k=3)

# 3. Verify and print results
print(f"Number of matches: {len(matches)}")

for i, doc in enumerate(matches):
    # Chroma returns 'Document' objects. 
    # The text is in .page_content and metadata is in .metadata
    print(f"\nMatch {i}:")
    print(f"Content: {doc.page_content[:500]}...") # Print first 500 characters
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Number of matches: 1

Match 0:
Content: Input:
I would like to seek expert advice regarding our companys upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?
Follow up question: What is the volume of the transactions?
Follow up question: What is the date of the transactions
F...
Source: Unknown


In [29]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# setup the llm
llm = ChatGroq(model= "llama-3.3-70b-versatile",
               temperature=0)

# advanced legal assistant prompt

legal_prompt_template = """You are an experienced lawyer specializing in extracting and interpreting information 
from legal documents to provide accurate advice.

GUIDELINES:
- Use the provided context to support your answer.
- Structure your response clearly and logically.
- If the answer is not available in the context, state 'I don't know' and explain why (e.g., insufficient context).
- Suggest potential research avenues if the context is incomplete.

CONTEXT:
{context}

QUESTION: 
{question}

PROFESSIONAL LEGAL RESPONSE:"""

prompt = ChatPromptTemplate.from_template(legal_prompt_template)

# 3. Helper to clean up retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# the RAG chain
# This automates the: Question -> Search -> Format -> Prompt -> LLM flow

context = vectorstore.as_retriever(search_kwargs={"k": 3})

legal_rag_chain = (
    # STEP A: Create the input data
    {"context": vectorstore.as_retriever(search_kwargs={"k": 3}) | format_docs, 
     "question": RunnablePassthrough()}
    
    # STEP B: Plug data into the instructions
    | prompt
    
    # STEP C: Send the filled-in prompt to the LLM (Groq)
    | llm
    
    # STEP D: Clean the result so it's just text
    | StrOutputParser()
)


# how to use it

# --- HOW TO USE IT ---
user_question = "What are the essential elements of a contract in English law?"
final_answer = legal_rag_chain.invoke(user_question)

print(f"QUESTION: {user_question}")
print("-" * 30)
print(f"AI LEGAL ADVICE:\n{final_answer}")

QUESTION: What are the essential elements of a contract in English law?
------------------------------
AI LEGAL ADVICE:
In English law, the essential elements of a contract are outlined in the Legal Statement on cryptoassets and smart contracts published by the UK Jurisdiction Taskforce (UKJT). According to Section 18 of the Legal Statement, a contract exists when:

1. **Two or more parties have reached an agreement**: The parties must have a meeting of minds on the terms of the contract.
2. **The parties intend to create a legal relationship**: The parties must have the intention to create a legally binding agreement.
3. **Each party has given something of benefit**: This is known as consideration, which can be a promise, an act, or a forbearance.

These elements are applicable to both traditional contracts and smart contracts. As stated in Section 18 of the Legal Statement, "a smart contract is capable of satisfying those requirements just as well as a more traditional or natural lan

In [36]:
def analyze_answer(answer: str, context: str):
    # We use .lower() to make the check more "forgiving"
    if any(chunk.lower() in answer.lower() for chunk in context.split("\n") if len(chunk) > 10):
        print("✅ Answer derived from database context.")
    else:
        print("⚠️ Answer likely generated by the language model.")

In [31]:
# 1. Search with relevance scores (normalized 0.0 to 1.0)
# 'results' will be a list of tuples: [(Document, Score), (Document, Score), ...]
results = vectorstore.similarity_search_with_relevance_scores(user_question, k=3)

# 2. Extract matches to mirror your 'matches' variable
# This matches your original 'matches' structure for the return statement
matches = results 

# 3. Determine confidence based on the highest score (the first result)
if matches:
    best_doc, best_score = matches[0]
    
    # Threshold check (0.7 - 0.9 is generally 'High Confidence' in Chroma)
    if best_score > 0.7:
        print(f"✅ High confidence in database-derived answer. (Score: {best_score:.2f})")
    else:
        print(f"⚠️ Answer may be more LLM-derived due to lower confidence. (Score: {best_score:.2f})")
else:
    print("❌ No matches found in the database.")

# Returns the list of (Document, Score) tuples
# Note: You access text via matches[0][0].page_content

⚠️ Answer may be more LLM-derived due to lower confidence. (Score: -0.10)


C:\Users\user\AppData\Local\Temp\ipykernel_25152\3966099715.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4dce7db0-5669-402d-bdad-8be5a1dfffe9', metadata={}, page_content='Input:\nI would like to seek expert advice regarding our companys upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?\nFollow up question: What is the volume of the transactions?\nFollow up question: What is the date of the transactions\nFollow up question: What was the price of purchased ETH? What was the value of ETH expressed in stablecoin?\nFollow up question: What is the stablecoin pegged to? \nFollow up question: Has this transaction will happen between dependent or independent parties?\nFollow up question: When is th

In [37]:
# 1. Manually get the text chunks first
retrieved_docs = vectorstore.as_retriever().invoke(user_question)
context_string = "\n".join(doc.page_content for doc in retrieved_docs)

# 2. Now the function will work because context_string is a STRING, not a Retriever
analyze_answer(final_answer.answer, context_string)

AttributeError: 'TextAccessor' object has no attribute 'answer'

In [35]:
# 1. Query Chroma (we use the relevance score version for safety)
results = vectorstore.similarity_search_with_relevance_scores(user_question, k=3)

# 2. Compile context (Chroma stores text in .page_content)
# We loop through the tuples, taking only the Document (doc) and ignoring the score (_)
retrieved_texts = [doc.page_content for doc, _score in results]

# 3. Join them into one clean string for the LLM
context = "\n\n".join(retrieved_texts)

# Verify
print(f"Context assembled! Total length: {len(context)} characters.")

Context assembled! Total length: 7427 characters.


C:\Users\user\AppData\Local\Temp\ipykernel_25152\2793042630.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4dce7db0-5669-402d-bdad-8be5a1dfffe9', metadata={}, page_content='Input:\nI would like to seek expert advice regarding our companys upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?\nFollow up question: What is the volume of the transactions?\nFollow up question: What is the date of the transactions\nFollow up question: What was the price of purchased ETH? What was the value of ETH expressed in stablecoin?\nFollow up question: What is the stablecoin pegged to? \nFollow up question: Has this transaction will happen between dependent or independent parties?\nFollow up question: When is th

In [25]:
# 1. Ask a question that relates to your legal document
query = "What are the essential elements of a contract in English law?"

# 2. Search ChromaDB for the most relevant chunk
docs = vectorstore.similarity_search(query, k=1)

# 3. Print the result
if docs:
    print(f"Retrieved Content:\n{docs[0].page_content}")
else:
    print("No relevant documents found.")

Retrieved Content:
Input:
I would like to seek expert advice regarding our companys upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?
Follow up question: What is the volume of the transactions?
Follow up question: What is the date of the transactions
Follow up question: What was the price of purchased ETH? What was the value of ETH expressed in stablecoin?
Follow up question: What is the stablecoin pegged to? 
Follow up question: Has this transaction will happen between dependent or independent parties?
Follow up question: When is the transaction planned for?
Follow up question: Is trading crypto the companys primary activity?
Input: 300 ETH converted to Tether, which is pegged to USD, transaction needs to go through by 1 A

In [22]:
query = "What are the essential elements of a contract in English law?"

# Create a retriever object
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Use it to get documents
relevant_docs = retriever.invoke(query)
relevant_docs

[Document(id='4dce7db0-5669-402d-bdad-8be5a1dfffe9', metadata={}, page_content='Input:\nI would like to seek expert advice regarding our companys upcoming transaction involving the disposal of cryptocurrency assets. Specifically, we are planning to sell a substantial amount of Ethereum (ETH) and convert it into stablecoins due to market volatility. Please highlight the tax implications of that. Do we need to formalise the sale of ETH with a formal agreement?\nFollow up question: What is the volume of the transactions?\nFollow up question: What is the date of the transactions\nFollow up question: What was the price of purchased ETH? What was the value of ETH expressed in stablecoin?\nFollow up question: What is the stablecoin pegged to? \nFollow up question: Has this transaction will happen between dependent or independent parties?\nFollow up question: When is the transaction planned for?\nFollow up question: Is trading crypto the companys primary activity?\nInput: 300 ETH converted to 